# 01 - VoxCeleb1 Exploration

## Objective

Explore VoxCeleb1 as a candidate dataset for speaker identity and privacy evaluation.

## Questions

- How many speakers are available?
- How many utterances are available per speaker?
- How many recordings are available per speaker?
- Are there enough independent recordings for train/validation/test splits?
- Can the English-only subset be identified reliably?

At this stage, we construct an English-only subset of VoxCeleb1 and keep only speakers with at least three videos.

The dataset is split at the **video level**, so that different utterances from the same video cannot appear in multiple splits. This avoids leakage between train, validation, and test sets.

The resulting utterance-level metadata is saved as:

`voxceleb_utterances.parquet`

In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/interview-anon/bin/python


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
DATA_DIR = Path.home() / "Documents" / "voxceleb1" / "wav"

print("Dataset path:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

Dataset path: /Users/juni/Documents/voxceleb1/wav
Exists: True


In [4]:
speakers = [p for p in DATA_DIR.iterdir() if p.is_dir()]

print("Number of speakers:", len(speakers))

Number of speakers: 1211


In [5]:
rows = []

for speaker_dir in speakers:
    video_dirs = [p for p in speaker_dir.iterdir() if p.is_dir()]
    wav_files = list(speaker_dir.rglob("*.wav"))

    rows.append({
        "speaker_id": speaker_dir.name,
        "n_videos": len(video_dirs),
        "n_utterances": len(wav_files)
    })

In [6]:
import pandas as pd

df = pd.DataFrame(rows)
df.head()

,speaker_id,n_videos,n_utterances
0,id10384,8,53
1,id11064,16,52
2,id10370,19,105
3,id11090,19,211
4,id10142,13,72


In [7]:
df.describe()

,n_videos,n_utterances
count,1211.000000,1211.000000
mean,18.018167,122.743187
std,5.981763,77.902902
min,8.000000,45.000000
25%,13.000000,69.000000
50%,17.000000,102.000000
75%,22.000000,148.000000
max,36.000000,1002.000000


In [8]:
df["n_videos"].sum()

np.int64(21820)

In [9]:
df["n_utterances"].sum()

np.int64(148642)

In [10]:
df.sort_values("n_utterances", ascending=False).head(20)

,speaker_id,n_videos,n_utterances
735,id10986,32,1002
183,id10096,16,656
974,id10020,27,589
291,id11111,23,570
647,id10710,26,489
1119,id10935,22,476
549,id11211,31,468
582,id10397,29,426
653,id10945,26,409
206,id10203,29,401


In [11]:
video_rows = []

for speaker_dir in speakers:
    for video_dir in speaker_dir.iterdir():
        if video_dir.is_dir():
            wav_files = list(video_dir.glob("*.wav"))

            video_rows.append({
                "speaker_id": speaker_dir.name,
                "video_id": video_dir.name,
                "n_utterances": len(wav_files)
            })

video_df = pd.DataFrame(video_rows)
video_df.head()

,speaker_id,video_id,n_utterances
0,id10384,vxBFGKGXSFA,7
1,id10384,qXG2i6-PxZY,15
2,id10384,t-zurNhQm8A,2
3,id10384,ozGl4eZyZ_E,3
4,id10384,Ay1xwkYsNAE,1


In [12]:
videos_per_speaker = (
    video_df.groupby("speaker_id")
    .size()
    .sort_values(ascending=False)
)

videos_per_speaker.describe()

count    1211.000000
mean       18.018167
std         5.981763
min         8.000000
25%        13.000000
50%        17.000000
75%        22.000000
max        36.000000
dtype: float64

In [13]:
from urllib.request import urlretrieve

METADATA_DIR = (
    Path.home()
    / "Documents"
    / "interview-anonymization"
    / "data"
    / "metadata"
)

METADATA_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGE_CSV_URL = (
    "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/"
    "data_workshop_2021/lang_vox1_final.csv"
)

LANGUAGE_CSV_PATH = METADATA_DIR / "lang_vox1_final.csv"

if not LANGUAGE_CSV_PATH.exists():
    print("Downloading language predictions...")
    urlretrieve(LANGUAGE_CSV_URL, LANGUAGE_CSV_PATH)

print("Language metadata:", LANGUAGE_CSV_PATH)

Language metadata: /Users/juni/Documents/interview-anonymization/data/metadata/lang_vox1_final.csv


In [14]:
language_df = pd.read_csv(LANGUAGE_CSV_PATH)

print(language_df.shape)
print(language_df.columns)
language_df.head(20)
language_df.sort_values(by="confidence").head(20)

(153516, 3)
Index(['filename', 'lang', 'confidence'], dtype='str')


,filename,lang,confidence
94718,id10788/XQvOFXeDr2A/00001.wav,Haitian,0.63
148136,id11209/WYujrLqr28Y/00001.wav,Tatar,0.65
148139,id11209/WYujrLqr28Y/00004.wav,Tatar,0.65
148138,id11209/WYujrLqr28Y/00003.wav,Tatar,0.65
148137,id11209/WYujrLqr28Y/00002.wav,Tatar,0.65
92889,id10774/8NPeiO5hD7g/00002.wav,Maori,0.67
92888,id10774/8NPeiO5hD7g/00001.wav,Maori,0.67
76480,id10635/GHMhG_ARHXo/00002.wav,Bosnian,0.67
76479,id10635/GHMhG_ARHXo/00001.wav,Bosnian,0.67
114363,id10946/bHMtGYB65rE/00007.wav,Nepali,0.69


In [15]:
print(video_df.columns)

Index(['speaker_id', 'video_id', 'n_utterances'], dtype='str')


In [16]:
normalized_filenames = (
    language_df["filename"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

path_parts = normalized_filenames.str.split("/")

language_df["speaker_id"] = path_parts.str[-3]
language_df["video_id"] = path_parts.str[-2]

language_df["language"] = (
    language_df["lang"]
    .astype(str)
    .str.strip()
    .str.lower()
)

language_df["language_confidence"] = pd.to_numeric(
    language_df["confidence"],
    errors="coerce"
)

language_df.head()

,filename,lang,confidence,speaker_id,video_id,language,language_confidence
0,id10001/1zcIwhmdeo4/00001.wav,English,0.99,id10001,1zcIwhmdeo4,english,0.99
1,id10001/1zcIwhmdeo4/00002.wav,English,0.99,id10001,1zcIwhmdeo4,english,0.99
2,id10001/1zcIwhmdeo4/00003.wav,English,0.99,id10001,1zcIwhmdeo4,english,0.99
3,id10001/7gWzIy6yIIk/00001.wav,English,0.98,id10001,7gWzIy6yIIk,english,0.98
4,id10001/7gWzIy6yIIk/00002.wav,English,0.98,id10001,7gWzIy6yIIk,english,0.98


In [17]:
language_video_df = (
    language_df
    .groupby(["speaker_id", "video_id"], as_index=False)
    .agg(
        language=("language", "first"),
        language_confidence=("language_confidence", "first")
    )
)

language_video_df.head()

,speaker_id,video_id,language,language_confidence
0,id10001,1zcIwhmdeo4,english,0.99
1,id10001,7gWzIy6yIIk,english,0.98
2,id10001,7w0IBEWc9Qw,english,0.98
3,id10001,9mQ11vBs1wc,english,0.93
4,id10001,DtdEYdViWdw,english,0.97


In [18]:
video_df = video_df.merge(
    language_video_df,
    on=["speaker_id", "video_id"],
    how="left"
)

video_df.head()

,speaker_id,video_id,n_utterances,language,language_confidence
0,id10384,vxBFGKGXSFA,7,korean,0.85
1,id10384,qXG2i6-PxZY,15,korean,0.82
2,id10384,t-zurNhQm8A,2,korean,0.81
3,id10384,ozGl4eZyZ_E,3,korean,0.82
4,id10384,Ay1xwkYsNAE,1,korean,0.80


In [19]:
video_df["language"].value_counts().head(20)

language
english       18486
welsh          1865
french          278
hindi           163
german          133
urdu             74
dutch            68
italian          63
maori            61
danish           60
spanish          54
afrikaans        39
nepali           27
persian          24
bosnian          21
portuguese       20
catalan          20
galician         19
assamese         16
swedish          16
Name: count, dtype: int64

In [20]:
min_confidences = language_df.groupby("lang")["confidence"].min()
print(min_confidences["English"])
video_df.loc[
    video_df["language"] == "english",
    "language_confidence"
].describe()

0.75


count    18486.000000
mean         0.963559
std          0.026638
min          0.750000
25%          0.960000
50%          0.970000
75%          0.980000
max          0.990000
Name: language_confidence, dtype: float64

In [21]:
video_df = video_df[
    video_df["language"] == "english"
].copy()

videos_per_speaker = (
    video_df.groupby("speaker_id")
    .size()
)

eligible_speakers = videos_per_speaker[
    videos_per_speaker >= 3
].index

video_df = video_df[
    video_df["speaker_id"].isin(eligible_speakers)
].copy()

print("Final English-only subset")
print("Speakers:", video_df["speaker_id"].nunique())
print("Videos:", len(video_df))
print("Utterances:", video_df["n_utterances"].sum())

Final English-only subset
Speakers: 1166
Videos: 18449
Utterances: 124843


In [22]:
videos_per_speaker = (
    video_df.groupby("speaker_id")
    .size()
    .sort_values(ascending=False)
)

In [23]:
rng = np.random.default_rng(42)

split_rows = []

for speaker_id, group in video_df.groupby("speaker_id"):
    videos = group["video_id"].tolist()

    rng.shuffle(videos)

    n = len(videos)

    # Target proportions
    n_train = int(round(0.70 * n))
    n_val = int(round(0.15 * n))
    n_test = n - n_train - n_val

    # Ensure at least one video in each split
    if n_val < 1:
        n_val = 1

    if n_test < 1:
        n_test = 1

    # Re-adjust train size if needed
    n_train = n - n_val - n_test

    train_videos = videos[:n_train]
    val_videos = videos[n_train:n_train + n_val]
    test_videos = videos[n_train + n_val:]

    for video_id in train_videos:
        split_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "split": "train"
        })

    for video_id in val_videos:
        split_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "split": "val"
        })

    for video_id in test_videos:
        split_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "split": "test"
        })

split_df = pd.DataFrame(split_rows)

In [24]:
split_counts = split_df["split"].value_counts()

print(split_counts)
print()
print(split_counts / split_counts.sum())

split
train    12834
val       2815
test      2800
Name: count, dtype: int64

split
train    0.695647
val      0.152583
test     0.151770
Name: count, dtype: float64


In [25]:
# Check that no video appears in multiple splits
duplicates = (
    split_df.groupby(["speaker_id", "video_id"])["split"]
    .nunique()
)

print("Videos appearing in multiple splits:",
      (duplicates > 1).sum())

Videos appearing in multiple splits: 0


In [26]:
# Number of speakers in each split
split_df.groupby("split")["speaker_id"].nunique()

split
test     1166
train    1166
val      1166
Name: speaker_id, dtype: int64

In [27]:
speaker_split_counts = (
    split_df.groupby("speaker_id")["split"]
    .nunique()
)

print(speaker_split_counts.value_counts())

split
3    1166
Name: count, dtype: int64


In [28]:
split_df = split_df.merge(
    video_df,
    on=["speaker_id", "video_id"],
    how="left"
)

In [29]:
split_utterance_counts = (
    split_df.groupby("split")["n_utterances"].sum()
)

split_utterance_counts / split_utterance_counts.sum()

split
test     0.147137
train    0.699655
val      0.153208
Name: n_utterances, dtype: float64

In [30]:
utterance_rows = []

for index, row in split_df.iterrows():
    speaker_id = row["speaker_id"]
    video_id = row["video_id"]
    split = row["split"]
    language = row["language"]
    language_confidence = row["language_confidence"]

    video_dir = DATA_DIR / speaker_id / video_id

    for wav_path in video_dir.glob("*.wav"):
        utterance_rows.append({
            "speaker_id": speaker_id,
            "video_id": video_id,
            "utterance_id": wav_path.stem,
            "path": str(wav_path),
            "split": split,
            "language": language,
            "language_confidence": language_confidence
        })

utterance_df = pd.DataFrame(utterance_rows)

In [31]:
utterance_df["language"].value_counts()

language
english    124843
Name: count, dtype: int64

In [32]:
assert set(utterance_df["language"].unique()) == {"english"}

In [33]:
utterance_df.head()

,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
1,id10001,1zcIwhmdeo4,00003,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
2,id10001,1zcIwhmdeo4,00002,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
3,id10001,7gWzIy6yIIk,00004,/Users/juni/Documents/voxceleb1/wav/id10001/7g...,train,english,0.98
4,id10001,7gWzIy6yIIk,00001,/Users/juni/Documents/voxceleb1/wav/id10001/7g...,train,english,0.98


In [34]:
utterance_df.shape

(124843, 7)

In [35]:
utterance_df["split"].value_counts()

split
train    87347
val      19127
test     18369
Name: count, dtype: int64

In [36]:
utterance_df.groupby("split")["speaker_id"].nunique()

split
test     1166
train    1166
val      1166
Name: speaker_id, dtype: int64

In [37]:
utterance_df.groupby(
    ["speaker_id", "video_id"]
)["split"].nunique().max()

np.int64(1)

In [38]:
METADATA_DIR = Path.home() / "Documents" / "interview-anonymization" / "data" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

utterance_df.to_parquet(
    METADATA_DIR / "voxceleb_utterances.parquet",
    index=False
)

print("Saved:", METADATA_DIR / "voxceleb_utterances.parquet")

Saved: /Users/juni/Documents/interview-anonymization/data/metadata/voxceleb_utterances.parquet
